In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Punjabi_Bagh_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,390.0,194.0,225.0,139.0,190.0,243.0,96.0,42.0,102.0,177.0,374.0,290.0
1,2,367.0,261.0,116.0,121.0,200.0,155.0,108.0,77.0,94.0,178.0,347.0,303.0
2,3,368.0,264.0,NaN,165.0,254.0,149.0,109.0,89.0,106.0,147.0,425.0,289.0
3,4,409.0,339.0,134.0,150.0,301.0,209.0,74.0,72.0,76.0,170.0,403.0,181.0
4,5,370.0,226.0,116.0,138.0,342.0,253.0,114.0,69.0,150.0,137.0,390.0,160.0
5,6,345.0,151.0,147.0,158.0,242.0,177.0,84.0,60.0,111.0,134.0,382.0,211.0
6,7,373.0,173.0,193.0,188.0,314.0,223.0,69.0,54.0,84.0,NaN,403.0,224.0
7,8,395.0,200.0,146.0,173.0,237.0,238.0,68.0,69.0,109.0,158.0,409.0,317.0
8,9,376.0,152.0,141.0,182.0,203.0,149.0,98.0,91.0,143.0,172.0,376.0,261.0
9,10,307.0,335.0,207.0,172.0,182.0,160.0,134.0,96.0,121.0,121.0,348.0,254.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,390.000000,194.000000,225.000000,139.000000,190.000000,243.000000,96.000000,42.0,102.000000,177.000000,374.000000,290.000000
1,2,367.000000,261.000000,116.000000,121.000000,200.000000,155.000000,108.000000,77.0,94.000000,178.000000,347.000000,303.000000
2,3,368.000000,264.000000,163.735294,165.000000,254.000000,149.000000,109.000000,89.0,106.000000,147.000000,425.000000,289.000000
3,4,409.000000,339.000000,134.000000,150.000000,301.000000,209.000000,74.000000,72.0,76.000000,170.000000,403.000000,181.000000
4,5,370.000000,226.000000,116.000000,138.000000,342.000000,253.000000,114.000000,69.0,150.000000,137.000000,390.000000,160.000000
5,6,345.000000,151.000000,147.000000,158.000000,242.000000,177.000000,84.000000,60.0,111.000000,134.000000,382.000000,211.000000
6,7,373.000000,173.000000,193.000000,188.000000,314.000000,223.000000,69.000000,54.0,84.000000,213.685714,403.000000,224.000000
7,8,395.000000,200.000000,146.000000,173.000000,237.000000,238.000000,99.617647,69.0,109.000000,158.000000,409.000000,317.000000
8,9,376.000000,152.000000,141.000000,182.000000,203.000000,149.000000,98.000000,91.0,143.000000,172.000000,376.000000,261.000000
9,10,307.000000,335.000000,207.000000,172.000000,182.000000,160.000000,134.000000,96.0,121.000000,121.000000,348.000000,254.000000
